In [13]:
import pandas as pd
import sqlite3

In [14]:
df = pd.read_csv("C:\Projects\LayoffLens\data\layoffs.csv")
df.shape

(4540, 11)

In [15]:
df['date'] = pd.to_datetime(df['date'],errors='coerce')
df['date_added'] = pd.to_datetime(df['date_added'],errors='coerce')
df['year'] = df['date'].dt.year
df['quarter'] = df['date'].dt.quarter
df['disclosed'] = df['total_laid_off'].notnull()

In [16]:
df['company'] = df['company'].str.strip().str.title()

In [17]:
# Split the data into two tables so they can be joined later in SQL:
# - companies: one row per company (industry, stage, location, funds raised)
# - layoffs: one row per layoff event (date, headcount, etc.)
# Companies get deduplicated since their info repeats across events;
# layoffs keep every row since each is a separate real event.

companies = df[['company','location','country','industry','stage','funds_raised']].drop_duplicates(subset='company')
layoffs = df[['company','date','total_laid_off','percentage_laid_off','source','date_added','year','quarter','disclosed']]

In [18]:
companies[companies['company'] == 'Appgate']

,company,location,country,industry,stage,funds_raised
2342,Appgate,Miami,United States,Security,Post-IPO,NaN


In [19]:
# save the two dataframes as tables in a SQLite database, so they can be queried with SQL
conn = sqlite3.connect('../sql/layoffs.db')
companies.to_sql('companies',conn,if_exists='replace',index=False)
layoffs.to_sql('layoffs',conn,if_exists='replace',index=False)
conn.close()
